# Master Model Comparison
## All Models — Multimodal Fraud Detection Thesis

This notebook produces the final comparison across all models trained in this thesis.
All multimodal-chapter models are evaluated on the same test set
(`mm_test_mixed_all_group_test.csv`) for a fair comparison.

### Structure
1. Dataset balance overview
2. Individual model results (metrics bar + confusion matrix + ROC curve)
3. Cross-model comparison — F1, ROC-AUC, Precision, Recall
4. Subgroup analysis — tab0_img0 and tab1_img0 F1 across models
5. Final summary table

## 0 · Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_curve, auc as sk_auc, precision_recall_curve

sys.path.insert(0, r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\src")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "data" / "processed"
NB_RESULTS   = PROJECT_ROOT / "notebook" / "results"
COMPARE_DIR  = NB_RESULTS / "master_comparison"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

# Consistent colour palette for all models
MODEL_COLOURS = {
    "LR (mm)":          "#8172B2",
    "XGBoost (mm)":     "#4C72B0",
    "ResNet-18":        "#DD8452",
    "EfficientNet-B0":  "#55A868",
    "Multimodal A":     "#C44E52",
    "Multimodal B":     "#937860",
    "Multimodal C":     "#DA8BC3",
    "Multimodal D":     "#8C8C8C",
}
print("Output dir:", COMPARE_DIR)

---
## 1 · Dataset balance overview

In [ ]:
test_df = pd.read_csv(DATA_DIR / "mm_test_mixed_all_group_test.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Label distribution
counts = test_df["final_label"].value_counts().sort_index()
bars = axes[0].bar(["Legitimate (0)", "Fraud (1)"], counts.values,
                   color=["#4C72B0","#DD8452"], edgecolor="white", width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 str(val), ha="center", va="bottom", fontsize=11)
axes[0].set_title(f"Test set label distribution  (n={len(test_df):,})")
axes[0].set_ylabel("Count"); axes[0].spines[["top","right"]].set_visible(False)

# Combo type distribution
COMBO_COLOURS = {"tab0_img0":"#4C72B0","tab1_img1":"#C44E52",
                 "tab1_img0":"#DD8452","tab0_img1":"#55A868"}
combo_counts = test_df["combo_type"].value_counts().sort_index()
clrs = [COMBO_COLOURS.get(c,"gray") for c in combo_counts.index]
bars2 = axes[1].bar(combo_counts.index, combo_counts.values,
                    color=clrs, edgecolor="white", width=0.5)
for bar, val in zip(bars2, combo_counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 str(val), ha="center", va="bottom", fontsize=11)
axes[1].set_title("Test set combo_type distribution")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=15)
axes[1].spines[["top","right"]].set_visible(False)

fig.suptitle("mm Mixed-All Test Set — Balance Overview", fontsize=13)
fig.tight_layout()
fig.savefig(COMPARE_DIR / "dataset_balance.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Label ratio 0:1 = 1:{counts.get(1,0)/max(counts.get(0,1),1):.2f}")
print("\nCombo type counts:")
for k, v in combo_counts.items():
    print(f"  {k}: {v} ({v/len(test_df)*100:.1f}%)")

---
## 2 · Individual model results helper

In [ ]:
def plot_model_results(model_name, y_true, y_prob, threshold,
                       save_path=None, colour="#C44E52"):
    """
    Consistent 3-panel plot: metrics bar + confusion matrix + ROC curve.
    Same style across all models for fair visual comparison.
    """
    y_pred = (y_prob >= threshold).astype(int)

    from sklearn.metrics import (accuracy_score, precision_score,
                                  recall_score, f1_score, roc_auc_score,
                                  confusion_matrix)
    metrics = {
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc":   float(roc_auc_score(y_true, y_prob)),
    }
    cm = confusion_matrix(y_true, y_pred)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Panel 1: Metrics bar
    metric_keys = ["accuracy","precision","recall","f1","roc_auc"]
    metric_vals = [metrics[k] for k in metric_keys]
    bar_colours = ["#4C72B0","#55A868","#DD8452","#C44E52","#8172B2"]
    bars = axes[0].bar(metric_keys, metric_vals,
                       color=bar_colours, edgecolor="white")
    for bar, val in zip(bars, metric_vals):
        axes[0].text(bar.get_x()+bar.get_width()/2,
                     bar.get_height()+0.005,
                     f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    axes[0].set_ylim(0, 1.12); axes[0].set_title("Test metrics")
    axes[0].spines[["top","right"]].set_visible(False)

    # Panel 2: Confusion matrix
    im = axes[1].imshow(cm, cmap="Blues")
    axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
    axes[1].set_xticklabels(["Pred Legitimate","Pred Fraud"])
    axes[1].set_yticklabels(["True Legitimate","True Fraud"])
    axes[1].set_title(f"Confusion matrix (t={threshold:.2f})")
    thresh_val = cm.max() / 2
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, str(cm[i,j]), ha="center", va="center",
                         fontsize=14, fontweight="bold",
                         color="white" if cm[i,j] > thresh_val else "black")

    # Panel 3: ROC curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = sk_auc(fpr, tpr)
    axes[2].plot(fpr, tpr, color=colour, lw=2,
                 label=f"Test ROC-AUC = {roc_auc:.4f}")
    axes[2].plot([0,1],[0,1],"k--",lw=1)
    axes[2].set_xlabel("False Positive Rate")
    axes[2].set_ylabel("True Positive Rate")
    axes[2].set_title("ROC Curve (test)")
    axes[2].legend(); axes[2].spines[["top","right"]].set_visible(False)

    fig.suptitle(model_name, fontsize=13)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return metrics


def load_pred_csv(path):
    """Load test_predictions.csv and return y_true, y_prob."""
    df = pd.read_csv(path)
    return df["y_true"].values, df["y_prob"].values

---
## 3 · LR tabular-only (best variant from notebook 17)

In [ ]:
# Best LR variant — load from saved predictions
# B_smote was best by F1 but collapsed (threshold=0.10, recall≈1.0)
# A_none has more balanced precision/recall
lr_variants = ["A_none","A_weight","A_smote","B_none","B_weight","B_smote"]
lr_results  = {}

for v in lr_variants:
    pred_path = NB_RESULTS / "mm_tabular_logreg" / v / f"{v}_threshold_results.csv"
    # Load joblib model predictions via saved threshold results
    # Use saved tuned metrics json instead
    metrics_path = NB_RESULTS / "mm_tabular_logreg" / v / f"{v}_tuned_test_metrics.json"
    cm_path      = NB_RESULTS / "mm_tabular_logreg" / v / f"{v}_tuned_test_confusion_matrix.csv"
    thr_path     = NB_RESULTS / "mm_tabular_logreg" / v / f"{v}_threshold_results.csv"

    if metrics_path.exists() and thr_path.exists():
        import json as _json
        with open(metrics_path) as f:
            m = _json.load(f)
        thr_df = pd.read_csv(thr_path)
        best_t = float(thr_df.loc[thr_df["f1"].idxmax(), "threshold"])
        lr_results[v] = {"metrics": m, "threshold": best_t}
        print(f"{v}: F1={m['f1']:.4f} Recall={m['recall']:.4f} t={best_t:.2f}")
    else:
        print(f"{v}: missing files")

# Pick best by F1 with recall < 0.99 (avoid collapsed variants)
best_lr = None
best_lr_f1 = 0
for v, r in lr_results.items():
    m = r["metrics"]
    if m["f1"] > best_lr_f1 and m["recall"] < 0.99:
        best_lr_f1 = m["f1"]
        best_lr = v

print(f"\nSelected best LR variant: {best_lr}")

In [ ]:
# Re-evaluate best LR from saved model predictions
# Since we need y_prob for ROC curve, reload from joblib
from tabular_baseline_utils import (
    build_preprocessor, build_logreg_pipeline,
    evaluate_model,
)
from mm_tabular_utils import (
    TARGET_COL, RAW_COLS, FE_COLS,
    NUMERIC_RAW, NUMERIC_FE, CAT_COLS,
    engineer_features,
)
import joblib

test_raw = pd.read_csv(DATA_DIR / "mm_test_mixed_all_group_test.csv")
test_fe  = engineer_features(test_raw)
y_test   = test_raw[TARGET_COL].values

# Load best LR model
lr_model_path = NB_RESULTS / "mm_tabular_logreg" / best_lr / f"{best_lr}_model.joblib"
lr_model = joblib.load(lr_model_path)

X_test_lr = test_raw[RAW_COLS] if best_lr.startswith("A") else test_fe[FE_COLS]
lr_probs  = lr_model.predict_proba(X_test_lr)[:, 1]
lr_thresh = lr_results[best_lr]["threshold"]

lr_metrics = plot_model_results(
    f"Logistic Regression — {best_lr} (tabular-only)",
    y_test, lr_probs, lr_thresh,
    save_path=COMPARE_DIR / "lr_best.png",
    colour="#8172B2"
)

## 4 · XGBoost tabular-only (best variant from notebook 18)

In [ ]:
xgb_variants = ["A_none","A_weight","A_smote","B_none","B_weight","B_smote"]
xgb_results  = {}

for v in xgb_variants:
    thr_path = NB_RESULTS / "mm_tabular_xgboost" / v / f"{v}_threshold_results.csv"
    if thr_path.exists():
        thr_df = pd.read_csv(thr_path)
        best_t = float(thr_df.loc[thr_df["f1"].idxmax(), "threshold"])
        best_f1 = float(thr_df.loc[thr_df["f1"].idxmax(), "f1"])
        best_rec = float(thr_df.loc[thr_df["f1"].idxmax(), "recall"])
        xgb_results[v] = {"threshold": best_t, "f1": best_f1, "recall": best_rec}
        print(f"{v}: F1={best_f1:.4f} Recall={best_rec:.4f} t={best_t:.2f}")

best_xgb = max(
    [(v,r) for v,r in xgb_results.items() if r["recall"] < 0.99],
    key=lambda x: x[1]["f1"], default=(None,None)
)[0]
if best_xgb is None:
    best_xgb = max(xgb_results.keys(), key=lambda v: xgb_results[v]["f1"])
print(f"\nSelected best XGBoost variant: {best_xgb}")

In [ ]:
from tabular_xgboost_utils import build_preprocessor as xgb_build_preprocessor

xgb_model_path = NB_RESULTS / "mm_tabular_xgboost" / best_xgb / f"{best_xgb}_model.joblib"
xgb_model = joblib.load(xgb_model_path)

X_test_xgb = test_raw[RAW_COLS] if best_xgb.startswith("A") else test_fe[FE_COLS]
xgb_probs  = xgb_model.predict_proba(X_test_xgb)[:, 1]
xgb_thresh = xgb_results[best_xgb]["threshold"]

xgb_metrics = plot_model_results(
    f"XGBoost — {best_xgb} (tabular-only)",
    y_test, xgb_probs, xgb_thresh,
    save_path=COMPARE_DIR / "xgboost_best.png",
    colour="#4C72B0"
)

## 5 · ResNet-18 image-only (best variant from notebook 15)

In [ ]:
# Load from saved test_predictions.csv
resnet_w_path  = NB_RESULTS / "mm_image_resnet18_weighted"  / "test_predictions.csv"
resnet_u_path  = NB_RESULTS / "mm_image_resnet18_unweighted" / "test_predictions.csv"

best_resnet_path = resnet_w_path if resnet_w_path.exists() else resnet_u_path
variant_label    = "Weighted" if resnet_w_path.exists() else "Unweighted"

y_true_r, y_prob_r = load_pred_csv(best_resnet_path)
thresh_r = float(pd.read_csv(
    best_resnet_path.parent / "threshold_sweep.csv"
).pipe(lambda d: d.loc[d["f1"].idxmax(), "threshold"]))

resnet_metrics = plot_model_results(
    f"ResNet-18 — {variant_label} (image-only)",
    y_true_r, y_prob_r, thresh_r,
    save_path=COMPARE_DIR / "resnet18_best.png",
    colour="#DD8452"
)

## 6 · EfficientNet-B0 image-only (best variant from notebook 16)

In [ ]:
effnet_w_path  = NB_RESULTS / "mm_image_efficientnet_weighted"  / "test_predictions.csv"
effnet_u_path  = NB_RESULTS / "mm_image_efficientnet_unweighted" / "test_predictions.csv"

best_effnet_path = effnet_w_path if effnet_w_path.exists() else effnet_u_path
variant_label_e  = "Weighted" if effnet_w_path.exists() else "Unweighted"

y_true_e, y_prob_e = load_pred_csv(best_effnet_path)
thresh_e = float(pd.read_csv(
    best_effnet_path.parent / "threshold_sweep.csv"
).pipe(lambda d: d.loc[d["f1"].idxmax(), "threshold"]))

effnet_metrics = plot_model_results(
    f"EfficientNet-B0 — {variant_label_e} (image-only)",
    y_true_e, y_prob_e, thresh_e,
    save_path=COMPARE_DIR / "efficientnet_best.png",
    colour="#55A868"
)

## 7 · Deep Multimodal (best experiment from notebook 19)

In [ ]:
mm_exp_dirs = {
    "A: Raw+Normal":   NB_RESULTS / "mm_deep_resnet18" / "A_raw_normal",
    "B: Raw+Sampler":  NB_RESULTS / "mm_deep_resnet18" / "B_raw_sampler",
    "C: FE+Normal":    NB_RESULTS / "mm_deep_resnet18" / "C_fe_normal",
    "D: FE+Sampler":   NB_RESULTS / "mm_deep_resnet18" / "D_fe_sampler",
}

mm_results = {}
for label, d in mm_exp_dirs.items():
    pred_path = d / "test_predictions.csv"
    sweep_path = d / "threshold_sweep.csv"
    if pred_path.exists() and sweep_path.exists():
        y_t, y_p = load_pred_csv(pred_path)
        thresh = float(pd.read_csv(sweep_path).pipe(
            lambda df: df.loc[df["f1"].idxmax(), "threshold"]))
        from sklearn.metrics import f1_score as _f1
        f1 = float(_f1(y_t, (y_p>=thresh).astype(int), zero_division=0))
        mm_results[label] = {"y_true":y_t,"y_prob":y_p,"threshold":thresh,"f1":f1}
        print(f"{label}: F1={f1:.4f} t={thresh:.2f}")
    else:
        print(f"{label}: not found")

best_mm_label = max(mm_results.keys(), key=lambda k: mm_results[k]["f1"])
print(f"\nBest multimodal experiment: {best_mm_label}")

In [ ]:
best_mm = mm_results[best_mm_label]
mm_metrics = plot_model_results(
    f"Deep Multimodal — {best_mm_label}",
    best_mm["y_true"], best_mm["y_prob"], best_mm["threshold"],
    save_path=COMPARE_DIR / "multimodal_best.png",
    colour="#C44E52"
)

---
## 8 · Cross-model comparison — all models on same test set

In [ ]:
# Build master comparison table
all_models = []

# LR
all_models.append({
    "model": "LR (mm)", "chapter": "Tabular",
    "threshold": lr_thresh,
    **{k: lr_metrics[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "y_true": y_test, "y_prob": lr_probs,
})

# XGBoost
all_models.append({
    "model": "XGBoost (mm)", "chapter": "Tabular",
    "threshold": xgb_thresh,
    **{k: xgb_metrics[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "y_true": y_test, "y_prob": xgb_probs,
})

# ResNet-18
all_models.append({
    "model": "ResNet-18", "chapter": "Image",
    "threshold": thresh_r,
    **{k: resnet_metrics[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "y_true": y_true_r, "y_prob": y_prob_r,
})

# EfficientNet
all_models.append({
    "model": "EfficientNet-B0", "chapter": "Image",
    "threshold": thresh_e,
    **{k: effnet_metrics[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "y_true": y_true_e, "y_prob": y_prob_e,
})

# Multimodal
all_models.append({
    "model": "Multimodal", "chapter": "Multimodal",
    "threshold": best_mm["threshold"],
    **{k: mm_metrics[k] for k in ["accuracy","precision","recall","f1","roc_auc"]},
    "y_true": best_mm["y_true"], "y_prob": best_mm["y_prob"],
})

cmp_df = pd.DataFrame([{k:v for k,v in m.items()
                         if k not in ["y_true","y_prob"]}
                        for m in all_models])
display(cmp_df.round(4))
cmp_df.to_csv(COMPARE_DIR / "master_comparison.csv", index=False)

### Cross-model metrics bar chart

In [ ]:
metrics_plot = ["accuracy","precision","recall","f1","roc_auc"]
metric_labels = ["Accuracy","Precision","Recall","F1","ROC-AUC"]
x = np.arange(len(metrics_plot)); width = 0.15
chapter_colours = {"Tabular":"#4C72B0","Image":"#DD8452","Multimodal":"#C44E52"}

fig, ax = plt.subplots(figsize=(14, 5))
for i, m in enumerate(all_models):
    colour = chapter_colours.get(m["chapter"], "gray")
    bars = ax.bar(x + i*width,
                  [m[metric] for metric in metrics_plot],
                  width, label=m["model"], color=colour,
                  alpha=0.7 + 0.1*(m["chapter"]=="Multimodal"),
                  edgecolor="white")

ax.set_xticks(x + width*2)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0.4, 1.12)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("All Models — Test Set Comparison (same test set)", fontsize=13)
ax.legend(fontsize=9, ncol=len(all_models))
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig(COMPARE_DIR / "metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### ROC curves — all models on same axes

In [ ]:
chapter_colours_roc = {
    "LR (mm)":         "#8172B2",
    "XGBoost (mm)":    "#4C72B0",
    "ResNet-18":       "#DD8452",
    "EfficientNet-B0": "#55A868",
    "Multimodal":      "#C44E52",
}
line_styles = {
    "LR (mm)": "--", "XGBoost (mm)": "-.",
    "ResNet-18": ":", "EfficientNet-B0": (0,(3,1,1,1)),
    "Multimodal": "-",
}

fig, ax = plt.subplots(figsize=(9, 7))
for m in all_models:
    fpr, tpr, _ = roc_curve(m["y_true"], m["y_prob"])
    roc_auc = sk_auc(fpr, tpr)
    ax.plot(fpr, tpr,
            color=chapter_colours_roc.get(m["model"],"gray"),
            linestyle=line_styles.get(m["model"],"-"),
            lw=2.5 if m["chapter"]=="Multimodal" else 1.8,
            label=f"{m['model']}  AUC={roc_auc:.4f}")

ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.5)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — All Models (same test set)", fontsize=13)
ax.legend(fontsize=10, loc="lower right")
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig(COMPARE_DIR / "roc_curves_all.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 9 · Subgroup analysis — combo_type F1 across models

In [ ]:
from sklearn.metrics import f1_score as _f1

COMBOS = ["tab0_img0","tab0_img1","tab1_img0","tab1_img1"]
COMBO_COLOURS = {"tab0_img0":"#4C72B0","tab1_img1":"#C44E52",
                 "tab1_img0":"#DD8452","tab0_img1":"#55A868"}

def combo_f1(y_true, y_prob, threshold, test_df_local):
    y_pred = (y_prob >= threshold).astype(int)
    pred_local = test_df_local.reset_index(drop=True).copy()
    pred_local["y_true"] = y_true
    pred_local["y_pred"] = y_pred
    result = {}
    for combo, g in pred_local.groupby("combo_type"):
        result[combo] = float(_f1(g["y_true"].values, g["y_pred"].values,
                                   zero_division=0))
    return result

sg_all = {}
for m in all_models:
    sg_all[m["model"]] = combo_f1(
        m["y_true"], m["y_prob"], m["threshold"], test_df
    )

sg_df = pd.DataFrame(sg_all).T[COMBOS]
display(sg_df.round(3))
sg_df.to_csv(COMPARE_DIR / "subgroup_comparison.csv")

In [ ]:
x = np.arange(len(COMBOS)); width = 0.15
model_names = [m["model"] for m in all_models]
model_colours_sg = [chapter_colours_roc.get(m["model"],"gray") for m in all_models]

fig, ax = plt.subplots(figsize=(14, 5))
for i, (mname, colour) in enumerate(zip(model_names, model_colours_sg)):
    vals = [sg_all[mname].get(c, 0) for c in COMBOS]
    bars = ax.bar(x + i*width, vals, width,
                  label=mname, color=colour,
                  alpha=0.7+(0.15 if mname=="Multimodal" else 0),
                  edgecolor="white")

ax.set_xticks(x + width*2)
ax.set_xticklabels([
    "tab0_img0\n(label 0, legit)",
    "tab0_img1\n(label 1, img fraud)",
    "tab1_img0\n(label 1, tab fraud)",
    "tab1_img1\n(label 1, both)",
], fontsize=9)
ax.axhline(0.5, color="red", linestyle="--", alpha=0.4, label="random")
ax.set_ylim(0, 1.2); ax.set_ylabel("F1 Score", fontsize=11)
ax.set_title("Subgroup F1 by combo_type — All Models", fontsize=13)
ax.legend(fontsize=9, ncol=len(all_models)+1)
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig(COMPARE_DIR / "subgroup_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nKey finding:")
print(f"  tab1_img0 F1 (image-only best):  {max(sg_all['ResNet-18'].get('tab1_img0',0), sg_all['EfficientNet-B0'].get('tab1_img0',0)):.3f}")
print(f"  tab1_img0 F1 (multimodal best):  {sg_all['Multimodal'].get('tab1_img0',0):.3f}")
print(f"  tab0_img0 F1 (multimodal best):  {sg_all['Multimodal'].get('tab0_img0',0):.3f}")

---
## 10 · Final summary table

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis("off")

col_labels = ["Model","Chapter","Threshold","Accuracy","Precision",
              "Recall","F1","ROC-AUC","tab0_img0 F1","tab1_img0 F1"]

cell_data = []
for m in all_models:
    sg = sg_all[m["model"]]
    cell_data.append([
        m["model"], m["chapter"],
        f"{m['threshold']:.2f}",
        f"{m['accuracy']:.4f}",
        f"{m['precision']:.4f}",
        f"{m['recall']:.4f}",
        f"{m['f1']:.4f}",
        f"{m['roc_auc']:.4f}",
        f"{sg.get('tab0_img0',float('nan')):.3f}",
        f"{sg.get('tab1_img0',float('nan')):.3f}",
    ])

tbl = ax.table(cellText=cell_data, colLabels=col_labels,
               cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.05, 2.0)

# Highlight header
for j in range(len(col_labels)):
    tbl[0,j].set_facecolor("#2c3e50")
    tbl[0,j].set_text_props(color="white", fontweight="bold")

# Highlight multimodal row
for i, m in enumerate(all_models):
    if m["chapter"] == "Multimodal":
        for j in range(len(col_labels)):
            tbl[i+1,j].set_facecolor("#ffeaa7")

ax.set_title("Master Model Comparison — mm Mixed-All Test Set",
             fontsize=13, pad=20)
fig.tight_layout()
fig.savefig(COMPARE_DIR / "final_summary_table.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nAll comparison plots saved to: {COMPARE_DIR}")